In [ ]:
# Download data

import requests
import zipfile
import io
import os
import pandas as pd

def download_retrosheet_data(years, data_type='gamelogs'):
    """
    Retrosheet adatok letöltése évek szerint
    """
    base_url = "http://www.retrosheet.org/"
    
    for year in years:
        try:
            if data_type == 'gamelogs':
                url = f"{base_url}gamelogs/gl{year}.zip"
                filename = f"gl{year}.txt"
            elif data_type == 'events':
                url = f"{base_url}events/{year}eve.zip"
                filename = f"{year}eve.zip"
            
            print(f"Letöltés: {year} {data_type}")
            response = requests.get(url)
            
            if response.status_code == 200:
                # ZIP kicsomagolása
                with zipfile.ZipFile(io.BytesIO(response.content)) as z:
                    z.extractall(f"retrosheet_data/{data_type}")
                print(f"Sikeres letöltés: {year}")
            else:
                print(f"Nem elérhető: {year}")
                
        except Exception as e:
            print(f"Hiba {year} letöltésénél: {e}")

# Évek letöltése
years = [2024, 2023, 2022, 2021, 2019]
download_retrosheet_data(years, 'gamelogs')

In [ ]:
# Parse data

import pandas as pd
import io

def parse_retrosheet_csv(csv_content):
    """
    Retrosheet CSV fájl feldolgozása DataFrame-mé
    """
    # Oszlopnevek a Retrosheet formátumhoz
    column_names = [
        'date', 'number_of_game', 'day_of_week', 'visiting_team', 'visiting_league',
        'visiting_team_game_number', 'home_team', 'home_league', 'home_team_game_number',
        'visiting_score', 'home_score', 'length_outs', 'day_night', 'completion_info',
        'forfeit_info', 'protest_info', 'park_id', 'attendance', 'time_of_game',
        'visiting_line_score', 'home_line_score', 'visiting_ab', 'visiting_h',
        'visiting_double', 'visiting_triple', 'visiting_hr', 'visiting_rbi',
        'visiting_sh', 'visiting_sf', 'visiting_hbp', 'visiting_bb', 'visiting_ibb',
        'visiting_so', 'visiting_sb', 'visiting_cs', 'visiting_gdp', 'visiting_ci',
        'visiting_lob', 'visiting_pitchers_used', 'visiting_individual_er',
        'visiting_team_er', 'visiting_wp', 'visiting_balks', 'visiting_putouts',
        'visiting_assists', 'visiting_errors', 'visiting_passed_balls',
        'visiting_double_plays', 'visiting_triple_plays', 'home_ab', 'home_h',
        'home_double', 'home_triple', 'home_hr', 'home_rbi', 'home_sh', 'home_sf',
        'home_hbp', 'home_bb', 'home_ibb', 'home_so', 'home_sb', 'home_cs', 'home_gdp',
        'home_ci', 'home_lob', 'home_pitchers_used', 'home_individual_er',
        'home_team_er', 'home_wp', 'home_balks', 'home_putouts', 'home_assists',
        'home_errors', 'home_passed_balls', 'home_double_plays', 'home_triple_plays',
        'home_plate_umpire', 'home_plate_umpire_id', 'first_base_umpire',
        'first_base_umpire_id', 'second_base_umpire', 'second_base_umpire_id',
        'third_base_umpire', 'third_base_umpire_id', 'left_field_umpire',
        'left_field_umpire_id', 'right_field_umpire', 'right_field_umpire_id',
        'visiting_manager', 'visiting_manager_id', 'home_manager', 'home_manager_id',
        'winning_pitcher', 'winning_pitcher_id', 'losing_pitcher', 'losing_pitcher_id',
        'saving_pitcher', 'saving_pitcher_id', 'game_winning_rbi', 'game_winning_rbi_id',
        'visiting_starting_pitcher', 'visiting_starting_pitcher_id',
        'home_starting_pitcher', 'home_starting_pitcher_id', 'visiting_player_1',
        'visiting_player_1_id', 'visiting_player_1_def_pos', 'visiting_player_2',
        'visiting_player_2_id', 'visiting_player_2_def_pos', 'visiting_player_3',
        'visiting_player_3_id', 'visiting_player_3_def_pos', 'visiting_player_4',
        'visiting_player_4_id', 'visiting_player_4_def_pos', 'visiting_player_5',
        'visiting_player_5_id', 'visiting_player_5_def_pos', 'visiting_player_6',
        'visiting_player_6_id', 'visiting_player_6_def_pos', 'visiting_player_7',
        'visiting_player_7_id', 'visiting_player_7_def_pos', 'visiting_player_8',
        'visiting_player_8_id', 'visiting_player_8_def_pos', 'visiting_player_9',
        'visiting_player_9_id', 'visiting_player_9_def_pos', 'home_player_1',
        'home_player_1_id', 'home_player_1_def_pos', 'home_player_2', 'home_player_2_id',
        'home_player_2_def_pos', 'home_player_3', 'home_player_3_id',
        'home_player_3_def_pos', 'home_player_4', 'home_player_4_id',
        'home_player_4_def_pos', 'home_player_5', 'home_player_5_id',
        'home_player_5_def_pos', 'home_player_6', 'home_player_6_id',
        'home_player_6_def_pos', 'home_player_7', 'home_player_7_id',
        'home_player_7_def_pos', 'home_player_8', 'home_player_8_id',
        'home_player_8_def_pos', 'home_player_9', 'home_player_9_id',
        'home_player_9_def_pos', 'additional_info', 'acquisition_info'
    ]
    
    # CSV beolvasása pandas-szal
    df = pd.read_csv(io.StringIO(csv_content), header=None, names=column_names, dtype=str)
    
    # Numerikus oszlopok konvertálása
    numeric_columns = [
        'visiting_score', 'home_score', 'length_outs', 'attendance', 'time_of_game',
        'visiting_ab', 'visiting_h', 'visiting_double', 'visiting_triple', 'visiting_hr',
        'visiting_rbi', 'visiting_sh', 'visiting_sf', 'visiting_hbp', 'visiting_bb',
        'visiting_ibb', 'visiting_so', 'visiting_sb', 'visiting_cs', 'visiting_gdp',
        'visiting_ci', 'visiting_lob', 'visiting_pitchers_used', 'visiting_individual_er',
        'visiting_team_er', 'visiting_wp', 'visiting_balks', 'visiting_putouts',
        'visiting_assists', 'visiting_errors', 'visiting_passed_balls',
        'visiting_double_plays', 'visiting_triple_plays', 'home_ab', 'home_h',
        'home_double', 'home_triple', 'home_hr', 'home_rbi', 'home_sh', 'home_sf',
        'home_hbp', 'home_bb', 'home_ibb', 'home_so', 'home_sb', 'home_cs', 'home_gdp',
        'home_ci', 'home_lob', 'home_pitchers_used', 'home_individual_er',
        'home_team_er', 'home_wp', 'home_balks', 'home_putouts', 'home_assists',
        'home_errors', 'home_passed_balls', 'home_double_plays', 'home_triple_plays'
    ]
    
    for col in numeric_columns:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    
    # Dátum oszlop konvertálása
    df['date'] = pd.to_datetime(df['date'], format='%Y%m%d', errors='coerce')
    
    # Üres értékek kezelése
    df = df.replace('"(none)"', None)
    df = df.replace('(none)', None)
    df = df.replace('', None)
    
    return df

csv_content = open("retrosheet_data//gamelogs/gl2023.txt").read()
df = parse_retrosheet_csv(csv_content)
for col in df.columns:
    print(col)

In [ ]:
# Multiple years data loading and combination

import pandas as pd
import numpy as np
from sklearn.model_selection import TimeSeriesSplit
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

def load_multiple_years_data(years=[2024, 2023, 2022, 2021, 2019]):
    """
    Load and combine multiple years of MLB data
    """
    all_dfs = []
    
    for year in years:
        try:
            csv_content = open(f"retrosheet_data/gamelogs/gl{year}.txt").read()
            df_year = parse_retrosheet_csv(csv_content)
            df_year['season'] = year
            all_dfs.append(df_year)
            print(f"Loaded {year}: {len(df_year)} games")
        except Exception as e:
            print(f"Could not load {year}: {e}")
    
    if all_dfs:
        combined_df = pd.concat(all_dfs, ignore_index=True)
        print(f"Total games loaded: {len(combined_df)}")
        return combined_df
    else:
        return pd.DataFrame()

# Load all years data
df_all = load_multiple_years_data()

In [ ]:
# Feature Engineering for Baseball

def create_baseball_features(df):
    """
    Create comprehensive baseball features for ML prediction
    """
    df = df.copy()
    
    # Basic data cleaning
    df = df.dropna(subset=['date', 'visiting_team', 'home_team', 'visiting_score', 'home_score'])
    df = df.sort_values('date')
    
    # Target variable: 0=Away win, 1=Home win
    df['target'] = (df['home_score'] > df['visiting_score']).astype(int)
    
    # Basic game stats per team
    df['home_runs_scored'] = df['home_score']
    df['home_runs_allowed'] = df['visiting_score']
    df['visiting_runs_scored'] = df['visiting_score'] 
    df['visiting_runs_allowed'] = df['home_score']
    
    # Offensive stats
    df['home_batting_avg'] = df['home_h'] / df['home_ab'].replace(0, 1)
    df['visiting_batting_avg'] = df['visiting_h'] / df['visiting_ab'].replace(0, 1)
    
    # Power stats
    df['home_slugging'] = (df['home_h'] + df['home_double'] + 2*df['home_triple'] + 3*df['home_hr']) / df['home_ab'].replace(0, 1)
    df['visiting_slugging'] = (df['visiting_h'] + df['visiting_double'] + 2*df['visiting_triple'] + 3*df['visiting_hr']) / df['visiting_ab'].replace(0, 1)
    
    # On-base percentage
    df['home_obp'] = (df['home_h'] + df['home_bb'] + df['home_hbp']) / (df['home_ab'] + df['home_bb'] + df['home_sf'] + df['home_hbp']).replace(0, 1)
    df['visiting_obp'] = (df['visiting_h'] + df['visiting_bb'] + df['visiting_hbp']) / (df['visiting_ab'] + df['visiting_bb'] + df['visiting_sf'] + df['visiting_hbp']).replace(0, 1)
    
    # Run differential
    df['home_run_diff'] = df['home_score'] - df['visiting_score']
    df['visiting_run_diff'] = df['visiting_score'] - df['home_score']
    
    # Pitching stats (ERA approximation)
    df['home_era_game'] = df['home_individual_er'] * 9 / (df['length_outs'] / 3).replace(0, 1)
    df['visiting_era_game'] = df['visiting_individual_er'] * 9 / (df['length_outs'] / 3).replace(0, 1)
    
    # Replace infinite values
    df = df.replace([np.inf, -np.inf], np.nan)
    
    return df

# Apply feature engineering
df_features = create_baseball_features(df_all)
print(f"Features created. Shape: {df_features.shape}")

In [ ]:
# Rolling Team Statistics

def create_team_rolling_stats(df):
    """
    Create rolling statistics for each team
    """
    all_team_stats = []
    
    # Get unique teams
    home_teams = df['home_team'].unique()
    visiting_teams = df['visiting_team'].unique()
    all_teams = set(list(home_teams) + list(visiting_teams))
    
    for team in all_teams:
        if pd.isna(team):
            continue
            
        try:
            # Get all games for this team (home and away)
            home_games = df[df['home_team'] == team].copy()
            away_games = df[df['visiting_team'] == team].copy()
            
            # Standardize columns
            home_games['team'] = team
            home_games['opponent'] = home_games['visiting_team']
            home_games['runs_for'] = home_games['home_score']
            home_games['runs_against'] = home_games['visiting_score']
            home_games['hits_for'] = home_games['home_h']
            home_games['hits_against'] = home_games['visiting_h']
            home_games['errors_for'] = home_games['home_errors']
            home_games['errors_against'] = home_games['visiting_errors']
            home_games['is_home'] = 1
            home_games['win'] = (home_games['home_score'] > home_games['visiting_score']).astype(int)
            
            away_games['team'] = team
            away_games['opponent'] = away_games['home_team']
            away_games['runs_for'] = away_games['visiting_score']
            away_games['runs_against'] = away_games['home_score']
            away_games['hits_for'] = away_games['visiting_h']
            away_games['hits_against'] = away_games['home_h']
            away_games['errors_for'] = away_games['visiting_errors']
            away_games['errors_against'] = away_games['home_errors']
            away_games['is_home'] = 0
            away_games['win'] = (away_games['visiting_score'] > away_games['home_score']).astype(int)
            
            # Combine and sort by date
            team_games = pd.concat([home_games, away_games]).sort_values('date')
            
            # Rolling statistics (last 10 and 20 games)
            for window in [10, 20]:
                team_games[f'wins_last_{window}'] = team_games['win'].rolling(window=window, min_periods=1).mean()
                team_games[f'runs_for_avg_{window}'] = team_games['runs_for'].rolling(window=window, min_periods=1).mean()
                team_games[f'runs_against_avg_{window}'] = team_games['runs_against'].rolling(window=window, min_periods=1).mean()
                team_games[f'run_diff_avg_{window}'] = (team_games['runs_for'] - team_games['runs_against']).rolling(window=window, min_periods=1).mean()
                team_games[f'hits_for_avg_{window}'] = team_games['hits_for'].rolling(window=window, min_periods=1).mean()
                team_games[f'hits_against_avg_{window}'] = team_games['hits_against'].rolling(window=window, min_periods=1).mean()
                team_games[f'errors_avg_{window}'] = team_games['errors_for'].rolling(window=window, min_periods=1).mean()
            
            all_team_stats.append(team_games)
            
        except Exception as e:
            print(f"Error processing team {team}: {e}")
            continue
    
    if all_team_stats:
        return pd.concat(all_team_stats)
    else:
        return pd.DataFrame()

# Create team rolling stats
print("Creating team rolling statistics...")
team_stats = create_team_rolling_stats(df_features)

if not team_stats.empty:
    print(f"Team stats created. Shape: {team_stats.shape}")
    
    # Merge rolling stats back to main dataframe
    # For home team stats
    home_stats = team_stats[team_stats['is_home'] == 1][
        ['date', 'team', 'wins_last_10', 'wins_last_20', 'runs_for_avg_10', 'runs_for_avg_20',
         'runs_against_avg_10', 'runs_against_avg_20', 'run_diff_avg_10', 'run_diff_avg_20',
         'hits_for_avg_10', 'hits_for_avg_20', 'hits_against_avg_10', 'hits_against_avg_20',
         'errors_avg_10', 'errors_avg_20']
    ]
    home_stats.columns = ['date', 'home_team'] + [f'home_{col}' for col in home_stats.columns[2:]]
    
    # For visiting team stats  
    away_stats = team_stats[team_stats['is_home'] == 0][
        ['date', 'team', 'wins_last_10', 'wins_last_20', 'runs_for_avg_10', 'runs_for_avg_20',
         'runs_against_avg_10', 'runs_against_avg_20', 'run_diff_avg_10', 'run_diff_avg_20',
         'hits_for_avg_10', 'hits_for_avg_20', 'hits_against_avg_10', 'hits_against_avg_20',
         'errors_avg_10', 'errors_avg_20']
    ]
    away_stats.columns = ['date', 'visiting_team'] + [f'visiting_{col}' for col in away_stats.columns[2:]]
    
    # Merge with main dataframe
    df_features = df_features.merge(home_stats, on=['date', 'home_team'], how='left')
    df_features = df_features.merge(away_stats, on=['date', 'visiting_team'], how='left')
    
    print(f"Final dataframe shape: {df_features.shape}")
else:
    print("Warning: Could not create team rolling statistics")

In [ ]:
# Additional Features

# Home field advantage
df_features['home_field_advantage'] = 1

# Day/Night game
df_features['is_day_game'] = (df_features['day_night'] == 'D').astype(int)

# Season timing (early, mid, late season)
df_features['day_of_year'] = df_features['date'].dt.dayofyear
df_features['season_phase'] = pd.cut(df_features['day_of_year'], 
                                   bins=[0, 120, 240, 366], 
                                   labels=[0, 1, 2], 
                                   include_lowest=True).astype(float)

# Rest days (approximate)
df_features = df_features.sort_values(['home_team', 'date'])
df_features['home_rest_days'] = df_features.groupby('home_team')['date'].diff().dt.days.fillna(5)

df_features = df_features.sort_values(['visiting_team', 'date'])  
df_features['visiting_rest_days'] = df_features.groupby('visiting_team')['date'].diff().dt.days.fillna(5)

# Fill missing values
numeric_cols = df_features.select_dtypes(include=[np.number]).columns
df_features[numeric_cols] = df_features[numeric_cols].fillna(0)

# Replace any remaining infinite values
df_features = df_features.replace([np.inf, -np.inf], 0)

print("Additional features created")
print(f"Available columns: {len(df_features.columns)}")

In [ ]:
# Model Building

# Select features for modeling
potential_features = [
    'home_batting_avg', 'visiting_batting_avg', 'home_slugging', 'visiting_slugging',
    'home_obp', 'visiting_obp', 'home_wins_last_10', 'visiting_wins_last_10',
    'home_wins_last_20', 'visiting_wins_last_20', 'home_runs_for_avg_10', 'visiting_runs_for_avg_10',
    'home_runs_for_avg_20', 'visiting_runs_for_avg_20', 'home_runs_against_avg_10', 'visiting_runs_against_avg_10',
    'home_runs_against_avg_20', 'visiting_runs_against_avg_20', 'home_run_diff_avg_10', 'visiting_run_diff_avg_10',
    'home_run_diff_avg_20', 'visiting_run_diff_avg_20', 'home_field_advantage', 'is_day_game',
    'season_phase', 'home_rest_days', 'visiting_rest_days', 'home_errors_avg_10', 'visiting_errors_avg_10'
]

# Filter to features that actually exist
feature_columns = [col for col in potential_features if col in df_features.columns]
print(f"Using {len(feature_columns)} features: {feature_columns}")

# Prepare modeling dataset
df_model = df_features.dropna(subset=feature_columns + ['target'])
df_model = df_model[df_model['target'].notna()]

print(f"Model dataset size: {len(df_model)} games")

if len(df_model) > 100:  # Minimum data check
    # Sort by date for time series split
    df_model = df_model.sort_values('date')
    X = df_model[feature_columns]
    y = df_model['target']
    
    # Data cleaning
    from scipy import stats
    
    # Remove extreme outliers
    for col in feature_columns:
        if X[col].dtype in [np.float64, np.float32]:
            z_scores = np.abs(stats.zscore(X[col]))
            outlier_mask = z_scores < 3
            X = X[outlier_mask]
            y = y[outlier_mask]
    
    print(f"After outlier removal: {len(X)} games")
    
    # TimeSeriesSplit
    tscv = TimeSeriesSplit(n_splits=5)
    splits = list(tscv.split(X))
    train_idx, test_idx = splits[-1]  # Use last split for final evaluation
    
    # Scale features
    scaler = RobustScaler()
    X_train_scaled = scaler.fit_transform(X.iloc[train_idx])
    X_test_scaled = scaler.transform(X.iloc[test_idx])
    
    X_train = pd.DataFrame(X_train_scaled, columns=feature_columns, index=X.iloc[train_idx].index)
    X_test = pd.DataFrame(X_test_scaled, columns=feature_columns, index=X.iloc[test_idx].index)
    
    y_train = y.iloc[train_idx]
    y_test = y.iloc[test_idx]
    
    print(f"Train: {len(X_train)}, Test: {len(X_test)}")
    
    # Models
    models = {
        'RandomForest': RandomForestClassifier(
            n_estimators=200, random_state=42, max_depth=10,
            min_samples_split=20, min_samples_leaf=10
        ),
        'GradientBoosting': GradientBoostingClassifier(
            n_estimators=200, random_state=42, max_depth=5,
            learning_rate=0.1, subsample=0.8
        ),
        'LogisticRegression': LogisticRegression(
            random_state=42, max_iter=1000, C=1.0
        )
    }
    
    # Train and evaluate models
    from sklearn.model_selection import cross_val_score
    
    trained_models = {}
    cv_scores = {}
    
    for name, model in models.items():
        try:
            print(f"\nTraining {name}...")
            
            # Cross-validation
            cv_accuracy = cross_val_score(model, X, y, cv=tscv, scoring='accuracy')
            cv_scores[name] = {
                'accuracy_mean': cv_accuracy.mean(),
                'accuracy_std': cv_accuracy.std(),
                'cv_scores': cv_accuracy
            }
            
            print(f"{name} CV Accuracy: {cv_accuracy.mean():.3f} (+/- {cv_accuracy.std() * 2:.3f})")
            
            # Train final model
            model.fit(X_train, y_train)
            
            # Test set evaluation
            y_pred = model.predict(X_test)
            test_accuracy = accuracy_score(y_test, y_pred)
            
            print(f"{name} Test Accuracy: {test_accuracy:.3f}")
            
            # Feature importance
            if hasattr(model, 'feature_importances_'):
                importance_df = pd.DataFrame({
                    'feature': feature_columns,
                    'importance': model.feature_importances_
                }).sort_values('importance', ascending=False)
                print(f"Top 5 features for {name}:")
                print(importance_df.head().to_string(index=False))
            
            trained_models[name] = model
            
        except Exception as e:
            print(f"Error training {name}: {e}")
    
    # Model comparison
    print("\n" + "="*60)
    print("MODEL COMPARISON SUMMARY")
    print("="*60)
    
    comparison_results = []
    for name, model in trained_models.items():
        y_pred = model.predict(X_test)
        y_proba = model.predict_proba(X_test)
        
        test_accuracy = accuracy_score(y_test, y_pred)
        cv_mean = cv_scores[name]['accuracy_mean']
        cv_std = cv_scores[name]['accuracy_std']
        
        comparison_results.append({
            'Model': name,
            'CV_Mean': cv_mean,
            'CV_Std': cv_std,
            'Test_Accuracy': test_accuracy,
            'Home_Win_Prob_Avg': y_proba[:, 1].mean()
        })
        
        # Detailed metrics
        print(f"\n{name} Classification Report:")
        print(classification_report(y_test, y_pred, target_names=['Away Win', 'Home Win']))
    
    # Display comparison
    comparison_df = pd.DataFrame(comparison_results)
    print("\nModel Performance Comparison:")
    print(comparison_df.round(3).to_string(index=False))
    
    # Visualizations
    plt.figure(figsize=(15, 10))
    
    # Confusion matrices
    for i, (name, model) in enumerate(trained_models.items()):
        plt.subplot(2, 3, i+1)
        y_pred = model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred)
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['Away Win', 'Home Win'],
                    yticklabels=['Away Win', 'Home Win'])
        plt.title(f'{name} Confusion Matrix')
    
    # Feature importance for tree-based models
    for i, (name, model) in enumerate(trained_models.items()):
        if hasattr(model, 'feature_importances_'):
            plt.subplot(2, 3, i+4)
            importance = model.feature_importances_
            indices = np.argsort(importance)[::-1][:10]
            
            plt.bar(range(10), importance[indices])
            plt.title(f'{name} Feature Importance (Top 10)')
            plt.xticks(range(10), [feature_columns[i] for i in indices], rotation=45)
    
    plt.tight_layout()
    plt.show()
    
    # Save models
    import joblib
    from datetime import datetime
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    for name, model in trained_models.items():
        joblib.dump(model, f'models/mlb_model_{name}_{timestamp}.joblib')
    
    joblib.dump(scaler, f'models/mlb_scaler_{timestamp}.joblib')
    joblib.dump(feature_columns, f'models/mlb_features_{timestamp}.joblib')
    
    print(f"\nModels saved with timestamp: {timestamp}")
    
    # Best model selection
    best_model_name = comparison_df.loc[comparison_df['Test_Accuracy'].idxmax(), 'Model']
    print(f"\nBest performing model: {best_model_name}")
    print(f"Test accuracy: {comparison_df.loc[comparison_df['Test_Accuracy'].idxmax(), 'Test_Accuracy']:.3f}")

else:
    print("Insufficient data for modeling")

print("\nMLB Predictor development completed!")

In [14]:
# Simplified Model using only Slugging, OBP and Batting Average
def create_simple_model(df):
    """
    Create MLB prediction model using only slugging, OBP and batting average
    """
    # Create a copy of the dataframe
    df_simple = df.copy()
    
    # Basic data cleaning
    df_simple = df_simple.dropna(subset=['date', 'visiting_team', 'home_team', 'visiting_score', 'home_score'])
    df_simple = df_simple.sort_values('date')
    
    # Target variable: 0=Away win, 1=Home win
    df_simple['target'] = (df_simple['home_score'] > df_simple['visiting_score']).astype(int)
    
    # Calculate the three key metrics for both teams
    # Batting average
    df_simple['home_batting_avg'] = df_simple['home_h'] / df_simple['home_ab'].replace(0, 1)
    df_simple['visiting_batting_avg'] = df_simple['visiting_h'] / df_simple['visiting_ab'].replace(0, 1)
    
    # Slugging percentage
    df_simple['home_slugging'] = (df_simple['home_h'] + df_simple['home_double'] + 
                                 2*df_simple['home_triple'] + 3*df_simple['home_hr']) / df_simple['home_ab'].replace(0, 1)
    df_simple['visiting_slugging'] = (df_simple['visiting_h'] + df_simple['visiting_double'] + 
                                    2*df_simple['visiting_triple'] + 3*df_simple['visiting_hr']) / df_simple['visiting_ab'].replace(0, 1)
    
    # On-base percentage
    df_simple['home_obp'] = (df_simple['home_h'] + df_simple['home_bb'] + df_simple['home_hbp']) / \
                           (df_simple['home_ab'] + df_simple['home_bb'] + df_simple['home_sf'] + df_simple['home_hbp']).replace(0, 1)
    df_simple['visiting_obp'] = (df_simple['visiting_h'] + df_simple['visiting_bb'] + df_simple['visiting_hbp']) / \
                               (df_simple['visiting_ab'] + df_simple['visiting_bb'] + df_simple['visiting_sf'] + df_simple['visiting_hbp']).replace(0, 1)
    
    # Replace infinite values
    df_simple = df_simple.replace([np.inf, -np.inf], np.nan)
    
    # Select only the 6 features we need (3 for home, 3 for away)
    simple_features = [
        'home_batting_avg', 'visiting_batting_avg',
        'home_slugging', 'visiting_slugging',
        'home_obp', 'visiting_obp'
    ]
    
    # Prepare modeling dataset
    df_simple_model = df_simple.dropna(subset=simple_features + ['target'])
    df_simple_model = df_simple_model[df_simple_model['target'].notna()]
    
    print(f"Simple model dataset size: {len(df_simple_model)} games")
    print(f"Features used: {simple_features}")
    
    if len(df_simple_model) > 100:
        # Sort by date for time series split
        df_simple_model = df_simple_model.sort_values('date')
        X_simple = df_simple_model[simple_features]
        y_simple = df_simple_model['target']
        
        # Remove extreme outliers
        from scipy import stats
        outlier_mask = np.ones(len(X_simple), dtype=bool)
        for col in simple_features:
            if X_simple[col].dtype in [np.float64, np.float32]:
                z_scores = np.abs(stats.zscore(X_simple[col]))
                outlier_mask = outlier_mask & (z_scores < 3)
        
        X_simple = X_simple[outlier_mask]
        y_simple = y_simple[outlier_mask]
        
        print(f"After outlier removal: {len(X_simple)} games")
        
        # TimeSeriesSplit
        tscv = TimeSeriesSplit(n_splits=5)
        splits = list(tscv.split(X_simple))
        train_idx, test_idx = splits[-1]  # Use last split for final evaluation
        
        # Scale features
        scaler_simple = RobustScaler()
        X_train_simple_scaled = scaler_simple.fit_transform(X_simple.iloc[train_idx])
        X_test_simple_scaled = scaler_simple.transform(X_simple.iloc[test_idx])
        
        X_train_simple = pd.DataFrame(X_train_simple_scaled, columns=simple_features, index=X_simple.iloc[train_idx].index)
        X_test_simple = pd.DataFrame(X_test_simple_scaled, columns=simple_features, index=X_simple.iloc[test_idx].index)
        
        y_train_simple = y_simple.iloc[train_idx]
        y_test_simple = y_simple.iloc[test_idx]
        
        print(f"Train: {len(X_train_simple)}, Test: {len(X_test_simple)}")
        
        # Models
        models_simple = {
            'RandomForest_Simple': RandomForestClassifier(
                n_estimators=100, random_state=42, max_depth=8,
                min_samples_split=15, min_samples_leaf=5
            ),
            'LogisticRegression_Simple': LogisticRegression(
                random_state=42, max_iter=1000, C=1.0
            ),
            'GradientBoosting_Simple': GradientBoostingClassifier(
                n_estimators=100, random_state=42, max_depth=4,
                learning_rate=0.1, subsample=0.8
            )
        }
        
        # Train and evaluate simple models
        trained_models_simple = {}
        
        for name, model in models_simple.items():
            try:
                print(f"\nTraining {name}...")
                
                # Cross-validation
                cv_accuracy = cross_val_score(model, X_simple, y_simple, cv=tscv, scoring='accuracy')
                print(f"{name} CV Accuracy: {cv_accuracy.mean():.3f} (+/- {cv_accuracy.std() * 2:.3f})")
                
                # Train final model
                model.fit(X_train_simple, y_train_simple)
                
                # Test set evaluation
                y_pred_simple = model.predict(X_test_simple)
                test_accuracy = accuracy_score(y_test_simple, y_pred_simple)
                
                print(f"{name} Test Accuracy: {test_accuracy:.3f}")
                
                # Feature importance
                if hasattr(model, 'feature_importances_'):
                    importance_df = pd.DataFrame({
                        'feature': simple_features,
                        'importance': model.feature_importances_
                    }).sort_values('importance', ascending=False)
                    print(f"Feature importance for {name}:")
                    print(importance_df.to_string(index=False))
                
                trained_models_simple[name] = model
                
            except Exception as e:
                print(f"Error training {name}: {e}")
        
        # Model comparison
        print("\n" + "="*60)
        print("SIMPLE MODEL COMPARISON (Only Slugging/OBP/BA)")
        print("="*60)
        
        comparison_results_simple = []
        for name, model in trained_models_simple.items():
            y_pred = model.predict(X_test_simple)
            y_proba = model.predict_proba(X_test_simple)
            
            test_accuracy = accuracy_score(y_test_simple, y_pred)
            
            comparison_results_simple.append({
                'Model': name,
                'Test_Accuracy': test_accuracy,
                'Home_Win_Prob_Avg': y_proba[:, 1].mean()
            })
            
            # Detailed metrics
            print(f"\n{name} Classification Report:")
            print(classification_report(y_test_simple, y_pred, target_names=['Away Win', 'Home Win']))
        
        # Display comparison
        comparison_df_simple = pd.DataFrame(comparison_results_simple)
        print("\nSimple Model Performance Comparison:")
        print(comparison_df_simple.round(3).to_string(index=False))
        
        # Save the best simple model
        best_simple_model_name = comparison_df_simple.loc[comparison_df_simple['Test_Accuracy'].idxmax(), 'Model']
        best_simple_model = trained_models_simple[best_simple_model_name]
        
        import joblib
        from datetime import datetime
        
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        joblib.dump(best_simple_model, f'models/mlb_simple_model_{best_simple_model_name}_{timestamp}.joblib')
        joblib.dump(scaler_simple, f'models/mlb_simple_scaler_{timestamp}.joblib')
        joblib.dump(simple_features, f'models/mlb_simple_features_{timestamp}.joblib')
        
        print(f"\nBest simple model ({best_simple_model_name}) saved with timestamp: {timestamp}")
        print(f"Test accuracy: {comparison_df_simple.loc[comparison_df_simple['Test_Accuracy'].idxmax(), 'Test_Accuracy']:.3f}")
        
        return best_simple_model, scaler_simple, simple_features
        
    else:
        print("Insufficient data for simple modeling")
        return None, None, None

# Run the simple model
print("Creating simplified model using only Slugging, OBP and Batting Average...")
best_simple_model, simple_scaler, simple_features = create_simple_model(df_features)

Creating simplified model using only Slugging, OBP and Batting Average...
Simple model dataset size: 126897 games
Features used: ['home_batting_avg', 'visiting_batting_avg', 'home_slugging', 'visiting_slugging', 'home_obp', 'visiting_obp']
After outlier removal: 124673 games
Train: 103895, Test: 20778

Training RandomForest_Simple...
RandomForest_Simple CV Accuracy: 0.877 (+/- 0.055)
RandomForest_Simple Test Accuracy: 0.909
Feature importance for RandomForest_Simple:
             feature  importance
   visiting_slugging    0.248923
        visiting_obp    0.197940
            home_obp    0.159802
       home_slugging    0.137215
visiting_batting_avg    0.128365
    home_batting_avg    0.127755

Training LogisticRegression_Simple...
LogisticRegression_Simple CV Accuracy: 0.850 (+/- 0.028)
LogisticRegression_Simple Test Accuracy: 0.872

Training GradientBoosting_Simple...
GradientBoosting_Simple CV Accuracy: 0.870 (+/- 0.069)
GradientBoosting_Simple Test Accuracy: 0.909
Feature importanc